<a href="https://colab.research.google.com/github/Mogi21/pytorch-deep-learning-labs/blob/main/01_pytorch_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

class CaliforniaHousingDataset(Dataset):
    """Custom PyTorch Dataset for loading and processing the California Housing data."""
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        # Ensure y is a float tensor and reshaped to (N, 1) for regression
        self.y = torch.tensor(y.values, dtype=torch.float32).view(-1, 1) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        return self.X[idx]

def load_and_preprocess_data(data_dir='task5'):
    """Loads CSV files, standardizes features, and creates PyTorch DataLoaders."""
    # 1. Load data
    X_full_df = pd.read_csv(f'{data_dir}/data_train.csv', index_col=0)
    y_full_df = pd.read_csv(f'{data_dir}/labels_train.csv', index_col=0)
    X_test_df = pd.read_csv(f'{data_dir}/data_test.csv', index_col=0)

    # 2. Train/Validation Split
    X_train_df, X_valid_df, y_train_df, y_valid_df = train_test_split(
        X_full_df, y_full_df, test_size=0.2, random_state=42
    )

    # 3. Standardize features (Critical for PyTorch Regression)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_df)
    X_valid_scaled = scaler.transform(X_valid_df)
    X_test_scaled = scaler.transform(X_test_df)

    # 4. Create DataLoaders
    train_dataset = CaliforniaHousingDataset(X_train_scaled, y_train_df)
    valid_dataset = CaliforniaHousingDataset(X_valid_scaled, y_valid_df)
    test_dataset = CaliforniaHousingDataset(X_test_scaled)

    loader_train = DataLoader(train_dataset, batch_size=64, shuffle=True)
    loader_valid = DataLoader(valid_dataset, batch_size=64, shuffle=False)
    loader_test = DataLoader(test_dataset, batch_size=64, shuffle=False)

    return loader_train, loader_valid, loader_test, X_train_scaled.shape[1]

In [ ]:
class RegressionNetwork(nn.Module):
    """
    A single-layer regression network.
    Outputs a single continuous value (House Price) instead of class logits.
    """
    def __init__(self, input_dim):
        super().__init__()
        self.layer = nn.Linear(input_dim, 1)

    def forward(self, x):
        return self.layer(x)

def train_regression(model, loader_train, loader_valid, epochs=50, lr=0.01):
    """Training infrastructure utilizing Mean Squared Error (MSE) for regression."""
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = model.to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    for epoch in range(epochs):
        model.train()
        train_losses = []
        for X_batch, y_batch in loader_train:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)

            optimizer.zero_grad()
            y_pred = model(X_batch)
            loss = loss_fn(y_pred, y_batch)

            loss.backward()
            optimizer.step()
            train_losses.append(loss.item())

        # Validation Phase
        model.eval()
        valid_losses = []
        with torch.no_grad():
            for X_batch, y_batch in loader_valid:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                y_pred = model(X_batch)
                valid_losses.append(loss_fn(y_pred, y_batch).item())

# Note: Execution call omitted for GitHub presentation to avoid file dependency errors.

In [ ]:
def generate_predictions(model, loader_test):
    """Generates final continuous predictions and formats them for submission."""
    device = next(model.parameters()).device
    model.eval()
    predictions = []

    with torch.no_grad():
        for X_batch in loader_test:
            X_batch = X_batch.to(device)
            y_pred = model(X_batch)
            predictions.extend(y_pred.cpu().numpy())

    # Convert to flat array and save
    predictions = np.array(predictions).flatten()
    submission_df = pd.DataFrame(predictions, columns=['predictions'])
    submission_df.to_csv('submission.csv')
    return submission_df